# Tutorial 1 — The challenge token as an option: a toy dynamic program

**Goal for you (Ayan):** by the end of this notebook you can (1) explain *win probability* and *leverage* in one sentence each, (2) explain what "the marginal value of a token" means, (3) derive the challenge rule *challenge iff p·g > (1−p)·MTV*, and (4) explain why keeping a successful challenge makes optimal play *more aggressive*, not less.

Everything here is a toy. Next week we replace every toy piece with the real thing (real win-probability tables from 11 seasons of games, real miss distances from Statcast, real 2026 challenges).

## 1. Three words

- **Win probability (WP):** given the situation (inning, outs, runners, score, count), the chance the home team wins. Estimated from history: of all games that were ever in this exact spot, how many did the home team win?
- **Leverage:** how much WP swings on the next event. Bottom 9th, tie, bases loaded, 3-2: enormous. Top 3rd, up 8: nothing.
- **Marginal token value (MTV):** how much better off you are, in future WP, holding *t* tokens instead of *t−1* — the value of the *option* to challenge later.

## 2. The decision at one pitch

You think a call was wrong with probability **p**. If it's overturned you gain **g** (in WP). If you challenge and win, you keep the token; if you lose, it's gone. So:

- challenge: `p·(g + V_keep) + (1−p)·V_lose`
- don't: `V_keep`

Challenge iff `p·g + p·V_keep + (1−p)·V_lose > V_keep`  ⇔  **`p·g > (1−p)·(V_keep − V_lose) = (1−p)·MTV`**.

Read that twice. If the token is worth little in the future (late in a blowout), you should challenge even when p is small. If it's worth a lot (early, close game), you need to be surer. That is the whole paper in one line.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)

# ---------- toy game generator ----------
# 18 half-innings. In each half-inning, 0-3 "borderline calls" arrive. Each has:
#   miss  : how far the pitch was outside/inside the zone in inches (positive = call was wrong in your favour)
#   lev   : leverage weight (late & close bigger); gain g = 0.02 * lev  (WP points)
#   p     : YOUR belief the call was wrong. Your perception is noisy: you see m = miss + noise, and p = Phi(m/sigma).
from scipy.stats import norm
SIGMA = 1.5   # inches of perception noise

def toy_game(seed):
    rng = np.random.default_rng(seed)
    opps = []
    score_close = rng.random() < 0.6           # 60% of games stay close
    for h in range(1, 19):
        n = rng.poisson(1.2)
        for _ in range(n):
            miss = rng.normal(-1.0, 2.5)        # most borderline calls are actually right (miss<0)
            lev = (0.5 + h/18) * (2.0 if score_close else 0.5) * rng.uniform(0.5, 1.5)
            m = miss + rng.normal(0, SIGMA)
            p = norm.cdf(m / SIGMA)
            opps.append(dict(h=h, g=0.02*lev, p=p, wrong=miss > 0))
    return opps

games = [toy_game(s) for s in range(20000)]
print("opportunities per game:", np.mean([len(g) for g in games]).round(2))

## 3. Three policies

- **Naive:** challenge whenever p ≥ 0.5.
- **Save-it:** never before the 7th inning, then naive.
- **DP-optimal:** solve backward for the value of holding tokens at each half-inning, then apply the rule above.

We evaluate all three on the same 20,000 toy games. A successful challenge banks g and keeps the token; a failed one loses it.

In [ ]:
def play(opps, policy, V=None):
    t, gain = 2, 0.0
    for o in opps:
        if t == 0:
            continue
        if policy == "naive":
            go = o["p"] >= 0.5
        elif policy == "save":
            go = o["h"] >= 13 and o["p"] >= 0.5
        elif policy == "dp":
            mtv = V[o["h"], t] - V[o["h"], t-1]      # value of this token vs one fewer, from now on (start-of-half approx)
            go = o["p"] * o["g"] > (1 - o["p"]) * mtv
        if go:
            if o["wrong"]:
                gain += o["g"]      # overturned: bank the gain, keep the token
            else:
                t -= 1              # upheld: token gone
    return gain

# ---------- backward induction for V[h, t] ----------
# V[h, t] = expected future gain from the START of half-inning h with t tokens, under optimal play.
# We estimate it from the toy games themselves: for each game and each h, run the within-half-inning DP with the
# continuation V[h+1, .], then average.
def solve_V(games, sweeps=2):
    V = np.zeros((20, 3))
    for _ in range(sweeps):
        for h in range(18, 0, -1):
            acc = np.zeros(3); n = 0
            for opps in games:
                W = V[h+1].copy()
                for o in reversed([o for o in opps if o["h"] == h]):
                    Wn = W.copy()
                    for t in (1, 2):
                        Wn[t] = max(W[t], o["p"]*(o["g"] + W[t]) + (1-o["p"])*W[t-1])
                    W = Wn
                acc += W; n += 1
            V[h] = acc / n
    return V

V = solve_V(games)
for pol in ["naive", "save", "dp"]:
    res = [play(g, pol, V) for g in games]
    print(f"{pol:6s}: average gain per game = {100*np.mean(res):.2f} WP points")

**Questions to answer in your own words:**
1. Why does *save-it* lose so badly even though it "protects" the tokens? (Hint: what is a token worth after the game ends?)
2. Look at `V[h, 2] - V[h, 1]` across h. Why does it fall as the game goes on?
3. Change `SIGMA` to 0.5 (a sharper-eyed catcher) and to 3.0 (a pitcher guessing). What happens to the DP's advantage over naive?

In [ ]:
mtv2 = V[1:19, 2] - V[1:19, 1]
mtv1 = V[1:19, 1] - V[1:19, 0]
plt.figure(figsize=(6,3.5))
plt.plot(range(1,19), 100*mtv1, label="1st token (t=1 vs 0)")
plt.plot(range(1,19), 100*mtv2, label="2nd token (t=2 vs 1)")
plt.xlabel("half-inning"); plt.ylabel("marginal token value (WP points)"); plt.legend(); plt.title("A token is worth less as the game runs out")
plt.tight_layout(); plt.show()

## 4. The threshold in inches (the thing a bench coach can use)

Because p = Φ(m/σ), the rule `p·g > (1−p)·MTV` becomes **`m > σ · Φ⁻¹( MTV / (g + MTV) )`**. So for each situation we can print "challenge if you think the pitch missed by more than X inches". When MTV is tiny relative to g, X is *negative*: challenge even if you think it was probably a strike — because the token is about to expire and it's free if you're right.

In [ ]:
def threshold_inches(g, mtv, sigma=SIGMA):
    q = np.clip(mtv/(g+mtv), 1e-9, 1-1e-9)
    return sigma * norm.ppf(q)

print("g = 2 WP points, sigma = 1.5 in")
for h in [1, 5, 9, 13, 17]:
    for t in (1, 2):
        mtv = V[h, t] - V[h, t-1]
        print(f"half-inning {h:2d}, tokens {t}: MTV={100*mtv:4.2f} pts -> challenge if perceived miss > {threshold_inches(0.02, mtv):+.2f} in")

## 5. What's real and what's toy here
| Toy | Real thing (next week) |
|---|---|
| `g = 0.02 × leverage` | ΔWP of flipping the call, from a WP model fit on 7.4M pitches (2015–2025) with terminal handling for walks/strikeouts |
| `miss ~ Normal(−1, 2.5)` | signed miss distance to the ABS zone from Statcast trajectories propagated to the plate midpoint |
| `SIGMA = 1.5` for everyone | σ estimated per role (catcher / batter / pitcher) from 2026 challenge outcomes by maximum likelihood |
| 0–3 opportunities per half-inning | every called pitch in real games, with real counts, outs, runners, and score |
| policies compared on toy games | actual 2026 team behaviour vs the DP-optimal policy: capture ratio, 9th-inning dump index, learning by month |

When you can explain each row of this table, you own the paper.